# Dynamic Thermodynamic Cost Calculation Engine

**Purpose:** Validates the dynamic cost formula using controlled synthetic TomTom telemetry inputs. The live Streamlit app will use real-time TomTom API values (`free_flow_time_sec`, `live_time_sec`) fed directly from the routing API response

### Key Mechanics
* **ELT Data Lake Ingestion:** Replaces static API calls with a localized `get_latest_fuel_data()` function that scans the `data/raw/` partition and dynamically loads the most recent timestamped JSON file (e.g., `raw_fuel_prices_2026-06-23.json`), ensuring cost logic always runs on the freshest web-scraped metrics.
* **GasWatchPH Schema Targeting:** Queries specific fuel arrays using the new standardized system `fuel_type_slug` (e.g., `"diesel"`) and executes inline regex cleaning (`.replace("₱", "")`) to convert string outputs into clean computational floats. 
* **Dynamic Traffic Multiplication:** Replaces legacy static `config.json` multipliers. It calculates physical engine idling penalties by dividing `free_flow_time_sec` by `live_time_sec`, generating a fluid efficiency degradation scale capped at `1.0`.
* **Extreme Gridlock Stress Testing:** Contains a hardcoded "Scenario B" designed strictly to stress-test the formula's boundaries. It forces a `0.50x` multiplier (simulating a drive that takes exactly twice as long as free-flow) to represent worst-case EDSA peak conditions, distinguishing it from actual light-traffic live metrics (e.g., `1.07x`).

In [3]:
import json
import os
import glob

# ==========================================
# 1. CORE COST CALCULATION ENGINE
# ==========================================
def calculate_dynamic_trip_cost(distance_km, vehicle_km_per_liter, free_flow_time_sec, live_time_sec, fuel_price_per_liter):
    """
    Computes total trip fuel cost by dynamically adjusting vehicle efficiency 
    using TomTom's real-time dynamic traffic load multiplier.
    """
    # Protect against divide-by-zero if live time is missing
    traffic_multiplier = (free_flow_time_sec / live_time_sec) if live_time_sec > 0 else 1.0
    
    # Cap the multiplier at 1.0 (traffic doesn't magically create fuel, even if driving faster than the limit)
    traffic_multiplier = min(traffic_multiplier, 1.0)

    actual_efficiency = vehicle_km_per_liter * traffic_multiplier
    liters_needed = distance_km / actual_efficiency
    total_cost = liters_needed * fuel_price_per_liter
    
    return total_cost, traffic_multiplier

# ==========================================
# 2. ELT DATA LAKE INGESTION
# ==========================================
def get_latest_fuel_data(directory=os.path.join("data", "raw")):
    """Scans the raw data lake partition and loads the most recent JSON payload."""
    list_of_files = glob.glob(os.path.join(directory, "*.json"))
    if not list_of_files:
        print("Error: No fuel data found. Please run the ELT extraction script first.")
        return None
    
    # OS-agnostic sort to grab the newest timestamped file
    latest_file = max(list_of_files, key=lambda f: os.path.basename(f))
    print(f"[System] Database Connected: Reading from {latest_file}")
    
    with open(latest_file, 'r', encoding='utf-8') as f:
        return json.load(f)

def get_specific_brand_price(fuel_data, target_fuel_slug, target_brand):
    """Searches the ELT JSON array for a specific brand's fuel price."""
    for record in fuel_data:
        # Aligned with the new GasWatchPH schema parameters
        if record.get("fuel_type_slug") == target_fuel_slug and record.get("brand") == target_brand:
            # Clean the new GasWatchPH currency format (e.g., "₱70.99")
            raw_price_str = record.get("current_price", "₱0")
            clean_price = float(raw_price_str.replace("₱", "").strip())
            return clean_price
            
    print(f"Error: Could not find {target_brand} offering {target_fuel_slug}.")
    return None

# ==========================================
# 3. STREAMLIT APP SIMULATION (TOMTOM INTEGRATED)
# ==========================================

# 1. Load the historical ledger on startup
scraped_data = get_latest_fuel_data()

if scraped_data:
    # 2. Simulate User Input from the UI Dropdowns
    user_fuel_choice = "diesel"  # Aligned to new programmatic slugs
    user_brand_choice = "Caltex"       
    
    # 3. Fetch the exact live price for that choice
    # NOTE FOR FINDING 1: This will extract the exact price staged in the loaded JSON file.
    live_price = get_specific_brand_price(scraped_data, user_fuel_choice, user_brand_choice)
    
    if live_price:
        print(f"\nLive Price Found: {user_brand_choice} {user_fuel_choice.title()} is ₱{live_price:.2f}/L")
        
        # 4. Simulate User Vehicle Input
        print("\n🚗 Welcome to the Route Optimizer")
        print("Helper: Check your dashboard trip computer for your exact km/L.")
        print("Helper: Most Metro Manila city cars average between 8 - 15 km/L.\n")

        raw_input = input("Enter your vehicle's fuel efficiency (km/L): ")
        clean_input = raw_input.lower().replace("km/l", "").replace("km", "").strip()

        try:
            vehicle_efficiency = float(clean_input)
            
            if vehicle_efficiency <= 0:
                print("\n🛑 ERROR: Efficiency must be greater than zero.")
            else:
                display_kml = int(vehicle_efficiency) if vehicle_efficiency.is_integer() else vehicle_efficiency
                print(f"\n✅ Vehicle locked in at {display_kml} km/L.")
                
                # ==========================================
                # 5. SIMULATE TOMTOM API TELEMETRY (STRESS TEST)
                # ==========================================
                trip_distance_km = 17.20 
                
                # Scenario A: Clear Roads (midnight, no traffic)
                # TomTom equivalent: live_time ≈ free_flow_time → multiplier = 1.00x
                clear_free_flow_sec = 34.2 * 60
                clear_live_sec = 34.2 * 60
                
                # Scenario B: Extreme Gridlock (EDSA peak, worst case stress test)
                # TomTom equivalent: live_time = 2x free_flow_time → multiplier = 0.50x
                # Note: Real TomTom data showed 1.07x for this route in light traffic.
                # This scenario demonstrates formula behavior under extreme conditions.
                rush_free_flow_sec = 34.2 * 60
                rush_live_sec = 68.4 * 60 
                
                # 6. Execute the dynamic formulas
                cost_clear, mult_clear = calculate_dynamic_trip_cost(trip_distance_km, vehicle_efficiency, clear_free_flow_sec, clear_live_sec, live_price)
                cost_rush, mult_rush = calculate_dynamic_trip_cost(trip_distance_km, vehicle_efficiency, rush_free_flow_sec, rush_live_sec, live_price)
                
                print("\n" + "=" * 50)
                print(f"📍 Route: SM North EDSA to Market! Market! ({trip_distance_km} km)")
                print(f"🚘 Vehicle Engine: User Baseline ({display_kml} km/L)")
                print("=" * 50)
                print(f"🟢 CLEAR TRAFFIC SCENARIO")
                print(f"   Multiplier: {mult_clear:.2f}x | Est. Cost: ₱{cost_clear:.2f}")
                print(f"🔴 RUSH HOUR SCENARIO (STRESS TEST)")
                print(f"   Multiplier: {mult_rush:.2f}x | Est. Cost: ₱{cost_rush:.2f}")
                print("=" * 50)

        except ValueError:
            print("\n🛑 ERROR: Please enter a valid number (e.g., 15 or 12.5). Do not use text.")

[System] Database Connected: Reading from data\raw\raw_fuel_prices_2026-06-23.json

Live Price Found: Caltex Diesel is ₱71.22/L

🚗 Welcome to the Route Optimizer
Helper: Check your dashboard trip computer for your exact km/L.
Helper: Most Metro Manila city cars average between 8 - 15 km/L.


✅ Vehicle locked in at 12 km/L.

📍 Route: SM North EDSA to Market! Market! (17.2 km)
🚘 Vehicle Engine: User Baseline (12 km/L)
🟢 CLEAR TRAFFIC SCENARIO
   Multiplier: 1.00x | Est. Cost: ₱102.08
🔴 RUSH HOUR SCENARIO (STRESS TEST)
   Multiplier: 0.50x | Est. Cost: ₱204.16
